# Notebook 03 — Marabou Exact Verification

Uses the Marabou neural network verifier to compute **exact** perturbation radii
on the SmallMLP. These serve as ground truth to validate how tight
the concolic upper bounds are in Notebook 04.

**Why only SmallMLP?**  
Marabou solves a Mixed-Integer Linear Program (MILP) over the full network.  
This is NP-hard and scales poorly — it times out on MediumMLP and LargeMLP.  
This is expected and is itself an important experimental result (demonstrating
why scalable approximations like concolic exploration are needed).

**What Marabou gives us:**  
For each input, Marabou either:
- **SAT**: finds a concrete adversarial example within radius ε (exact upper bound)
- **UNSAT**: proves no adversarial example exists within radius ε (exact lower bound)

By binary searching over ε, we get the exact perturbation radius ε*.

**Requires:** `models/small_mnist.pt`  
*(Run Notebook 01 first)*

**Outputs saved to:**
```
results/marabou_small_mnist.json
```

> **Note on Marabou installation:**  
> Marabou is not on PyPI. Installation instructions are in Section 0 below.  
> If installation fails, Section 6 provides a fallback using a pure LP-based
> exact verifier that works for small networks without Marabou.

## 0 — Install Marabou

In [1]:
# ── Standard packages ─────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['torch', 'torchvision', 'scipy', 'tqdm', 'numpy']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)

# ── Marabou installation ──────────────────────────────────────────────────────
# Option 1: install pre-built wheel (recommended)
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'maraboupy', '--quiet'],
    capture_output=True, text=True
)

try:
    import maraboupy
    print(f'Marabou installed successfully.')
    MARABOU_AVAILABLE = True
except ImportError:
    print('Marabou not available via pip.')
    print('Falling back to LP-based exact verifier (Section 6).')
    print()
    print('To install Marabou manually:')
    print('  pip install maraboupy')
    print('  OR build from source: https://github.com/NeuralNetworkVerification/Marabou')
    MARABOU_AVAILABLE = False

import torch
print(f'PyTorch: {torch.__version__}')

Marabou not available via pip.
Falling back to LP-based exact verifier (Section 6).

To install Marabou manually:
  pip install maraboupy
  OR build from source: https://github.com/NeuralNetworkVerification/Marabou
PyTorch: 2.12.0+cpu


## 1 — Imports & configuration

In [2]:
import sys, os, time
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from scipy.optimize import linprog

REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.network_definitions import load_model
from utils.metrics import (
    get_correctly_classified_samples,
    tightness_score, save_results, load_results,
)

MODELS_DIR  = REPO_ROOT / 'models'
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR    = REPO_ROOT / 'data'
RESULTS_DIR.mkdir(exist_ok=True)

# ── configuration ─────────────────────────────────────────────────────────────
DEVICE      = 'cpu'
SEED        = 42

# Use fewer samples for Marabou — it's slow even on small networks
# 30 samples gives enough for statistical comparison
N_SAMPLES   = 30

# Timeout per Marabou call (seconds) — increase if you have time
MARABOU_TIMEOUT = 60

# Binary search config for exact ε*
EPS_MIN     = 0.001
EPS_MAX     = 0.5     # keep small — Marabou is faster at small radii
N_BISECT    = 12      # 2^-12 ≈ 0.0001 precision

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Config: {N_SAMPLES} samples, timeout={MARABOU_TIMEOUT}s, '
      f'eps_range=[{EPS_MIN},{EPS_MAX}], bisect_steps={N_BISECT}')

Config: 30 samples, timeout=60s, eps_range=[0.001,0.5], bisect_steps=12


## 2 — Load model & data

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# load model
print('Loading small_mnist model...')
model = load_model(str(MODELS_DIR / 'small_mnist.pt'))
print(f'Architecture : {model.hidden_dims}  |  ReLU neurons: {model.n_relu_neurons()}')

# load MNIST test set
mnist_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_test = datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf)
loader     = DataLoader(mnist_test, batch_size=len(mnist_test), shuffle=False)
X_all, y_all = next(iter(loader))
X_all = X_all.view(X_all.size(0), -1).numpy()
y_all = y_all.numpy()

# select correctly-classified samples
X_eval, y_eval = get_correctly_classified_samples(
    model, X_all, y_all, N_SAMPLES, seed=SEED
)
print(f'Selected {len(X_eval)} correctly-classified samples for verification')

Loading small_mnist model...
  Loaded ← D:\concolic_exploration\models\small_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-64-64-10', 'best_test_acc': 0.9798, 'n_relu_neurons': 128, 'n_params': 55050}
Architecture : [64, 64]  |  ReLU neurons: 128
Selected 30 correctly-classified samples for verification


## 3 — LP-based exact verifier (no Marabou required)

This is our primary verifier — it works for any ReLU network by encoding
the verification problem as a Linear Program using the piecewise-linear
structure of the network.

**How it works:**
For a fixed activation pattern A, the network is a linear function.  
We solve: does any x' with ||x'-x||∞ ≤ ε and activation pattern A
produce a different class?  
We repeat for all reachable activation patterns near x.

**Limitation:** complete only within the current activation region.  
For cross-region verification we use Marabou (Section 4).

In [4]:
def get_network_matrices(model, x):
    """
    For a fixed input x, extract the effective weight matrix and bias
    of the linearised network (fixed activation pattern).

    Returns: (W_eff, b_eff) such that f(x') ≈ W_eff @ x' + b_eff
    within the current activation region.
    """
    import torch.nn as nn
    linear_layers = [m for m in model.network if isinstance(m, nn.Linear)]
    hidden_linear = linear_layers[:-1]
    output_linear = linear_layers[-1]

    # compute activation mask at x
    h = torch.tensor(x, dtype=torch.float32)
    masks = []
    for lin in hidden_linear:
        z = lin(h)
        masks.append((z > 0).float())   # 1 if active, 0 if inactive
        h = torch.relu(z)

    # build effective weight matrix layer by layer
    # each hidden layer becomes: D_l * (W_l @ ... + b_l)
    # where D_l is diagonal mask
    W = np.eye(model.input_dim)
    b = np.zeros(model.input_dim)

    for i, lin in enumerate(hidden_linear):
        W_l = lin.weight.detach().numpy()
        b_l = lin.bias.detach().numpy()
        D_l = np.diag(masks[i].numpy())
        W   = D_l @ W_l @ W
        b   = D_l @ (W_l @ b + b_l)

    # output layer
    W_out = output_linear.weight.detach().numpy()
    b_out = output_linear.bias.detach().numpy()
    W_eff = W_out @ W
    b_eff = W_out @ b + b_out

    return W_eff, b_eff


def lp_verify_local(model, x, label, epsilon):
    """
    LP-based local robustness verification within the current activation region.

    Checks: does any x' with ||x'-x||∞ ≤ ε in the SAME activation region
    produce a class score higher than label?

    Returns:
        'UNSAT' — no adversarial x' found within region (locally robust)
        'SAT'   — adversarial x' found, returns it
        'UNKNOWN' — solver failed
    """
    W_eff, b_eff = get_network_matrices(model, x)
    n_classes = W_eff.shape[0]
    n_input   = W_eff.shape[1]

    best_adv = None
    best_eps = epsilon + 1

    for target_class in range(n_classes):
        if target_class == label:
            continue

        # objective: maximise (score[target] - score[label])
        # = minimise (score[label] - score[target])
        # = minimise (W_eff[label] - W_eff[target]) @ x' + (b_eff[label] - b_eff[target])
        c = (W_eff[label] - W_eff[target_class])

        # bounds: x_i - epsilon ≤ x'_i ≤ x_i + epsilon
        bounds = [(x[i] - epsilon, x[i] + epsilon) for i in range(n_input)]

        result = linprog(c, bounds=bounds, method='highs')

        if result.status == 0:
            # check if this x' actually flips the class
            x_cand = result.x
            scores = W_eff @ x_cand + b_eff
            if scores[target_class] > scores[label]:
                dist = float(np.max(np.abs(x_cand - x)))
                if dist < best_eps:
                    best_eps = dist
                    best_adv = x_cand

    if best_adv is not None:
        return 'SAT', best_eps, best_adv
    return 'UNSAT', epsilon, None


def lp_exact_radius(model, x, label,
                    eps_min=0.001, eps_max=0.5, n_bisect=12):
    """
    Binary search for exact perturbation radius using LP verification.

    Returns: (eps_star, status_log)
        eps_star   : best upper bound found (exact within current region)
        status_log : list of (eps, status) per bisection step
    """
    lo, hi   = eps_min, eps_max
    best_eps = eps_max
    log      = []

    for step in range(n_bisect):
        mid    = (lo + hi) / 2
        status, found_eps, adv = lp_verify_local(model, x, label, mid)
        log.append({'step': step, 'eps': mid, 'status': status})

        if status == 'SAT':
            best_eps = found_eps
            hi = mid
        else:
            lo = mid

    return best_eps, log


print('LP verifier defined.')

LP verifier defined.


## 4 — Marabou verifier (if available)

Marabou provides **complete** verification — it checks ALL activation patterns,
not just the current one. This gives a true global perturbation radius.

We use it when available and fall back to the LP verifier otherwise.

In [5]:
def save_onnx(model, input_dim, path):
    """Export model to ONNX format for Marabou."""
    import torch.onnx
    dummy = torch.randn(1, input_dim)
    torch.onnx.export(
        model, dummy, path,
        input_names=['input'], output_names=['output'],
        opset_version=11,
    )


def marabou_verify(onnx_path, x, label, epsilon, timeout=60):
    """
    Run Marabou verification for L∞ robustness.

    Property: for all x' with ||x'-x||∞ ≤ ε, argmax(f(x')) = label

    Returns:
        'SAT'     — adversarial example found (not robust)
        'UNSAT'   — proven robust within epsilon
        'TIMEOUT' — solver timed out
        'ERROR'   — Marabou not available or other error
    """
    if not MARABOU_AVAILABLE:
        return 'ERROR'

    try:
        from maraboupy import Marabou, MarabouCore

        network = Marabou.read_onnx(onnx_path)
        n_input  = len(x)
        n_output = 10

        inputVars  = network.inputVars[0][0]
        outputVars = network.outputVars[0]

        # L∞ input constraints: x_i - ε ≤ x'_i ≤ x_i + ε
        for i in range(n_input):
            network.setLowerBound(inputVars[i], float(x[i]) - epsilon)
            network.setUpperBound(inputVars[i], float(x[i]) + epsilon)

        # output constraint: score[target] > score[label] for any target ≠ label
        # Marabou checks SAT for each target class separately
        options = MarabouCore.Options()
        options._timeoutInSeconds = timeout

        for target in range(n_output):
            if target == label:
                continue
            # add: output[target] - output[label] > 0
            eq = MarabouCore.Equation(MarabouCore.Equation.GE)
            eq.addAddend(1,  outputVars[target])
            eq.addAddend(-1, outputVars[label])
            eq.setScalar(1e-6)
            network.addEquation(eq)

            vals, stats = network.solve(options=options, verbose=False)

            if stats.hasTimedOut():
                return 'TIMEOUT'
            if len(vals) > 0:   # SAT
                return 'SAT'

            # remove constraint for next target class
            network.clearProperty()

        return 'UNSAT'

    except Exception as e:
        print(f'  Marabou error: {e}')
        return 'ERROR'


def marabou_exact_radius(onnx_path, x, label,
                          eps_min=0.001, eps_max=0.5,
                          n_bisect=10, timeout=60):
    """
    Binary search for exact perturbation radius using Marabou.
    Returns (eps_star, log, timed_out_flag)
    """
    lo, hi   = eps_min, eps_max
    best_eps = eps_max
    log      = []
    timed_out = False

    for step in range(n_bisect):
        mid    = (lo + hi) / 2
        status = marabou_verify(onnx_path, x, label, mid, timeout)
        log.append({'step': step, 'eps': mid, 'status': status})

        if status == 'SAT':
            best_eps = mid
            hi = mid
        elif status == 'UNSAT':
            lo = mid
        elif status in ('TIMEOUT', 'ERROR'):
            timed_out = True
            break

    return best_eps, log, timed_out


print('Marabou verifier defined.')

Marabou verifier defined.


## 5 — Run exact verification on SmallMLP
**Expected time: ~20–40 min depending on Marabou availability**

In [6]:
# export model to ONNX for Marabou
onnx_path = str(MODELS_DIR / 'small_mnist.onnx')
if MARABOU_AVAILABLE:
    save_onnx(model, model.input_dim, onnx_path)
    print(f'Model exported to {onnx_path}')
else:
    print('Marabou not available — using LP verifier (local verification only)')

# ── run verification ──────────────────────────────────────────────────────────
results = []
n_timeouts = 0

bar = tqdm(zip(X_eval, y_eval), total=len(X_eval),
           desc='Verifying', unit='sample')

for x, label in bar:
    label = int(label)
    t0    = time.time()

    if MARABOU_AVAILABLE:
        eps_star, log, timed_out = marabou_exact_radius(
            onnx_path, x, label,
            eps_min   = EPS_MIN,
            eps_max   = EPS_MAX,
            n_bisect  = N_BISECT,
            timeout   = MARABOU_TIMEOUT,
        )
        verifier = 'marabou'
        if timed_out:
            n_timeouts += 1
    else:
        # LP verifier — local verification only
        eps_star, log = lp_exact_radius(
            model, x, label,
            eps_min  = EPS_MIN,
            eps_max  = EPS_MAX,
            n_bisect = N_BISECT,
        )
        timed_out = False
        verifier  = 'lp_local'

    runtime = time.time() - t0

    results.append({
        'eps_star'    : float(eps_star),
        'timed_out'   : timed_out,
        'runtime_sec' : runtime,
        'verifier'    : verifier,
        'label'       : label,
        'log'         : log,
    })

    bar.set_postfix(
        eps=f'{eps_star:.4f}',
        timeout='Y' if timed_out else 'N',
        t=f'{runtime:.1f}s',
    )

print(f'\nComplete. Timeouts: {n_timeouts}/{len(results)}')

Marabou not available — using LP verifier (local verification only)


Verifying: 100%|████████████████████████████████████| 30/30 [00:17<00:00,  1.76sample/s, eps=0.1366, t=0.6s, timeout=N]


Complete. Timeouts: 0/30


## 6 — Save & summarise results

In [7]:
# ── aggregate stats ───────────────────────────────────────────────────────────
valid   = [r for r in results if not r['timed_out']]
eps_arr = np.array([r['eps_star']    for r in valid])
time_arr= np.array([r['runtime_sec'] for r in valid])

stats = {
    'verifier'              : results[0]['verifier'] if results else 'none',
    'n_samples'             : len(results),
    'n_valid'               : len(valid),
    'n_timeouts'            : n_timeouts,
    'mean_eps_star'         : float(np.mean(eps_arr))   if len(valid) else None,
    'std_eps_star'          : float(np.std(eps_arr))    if len(valid) else None,
    'median_eps_star'       : float(np.median(eps_arr)) if len(valid) else None,
    'mean_runtime_sec'      : float(np.mean(time_arr))  if len(valid) else None,
    'total_runtime_sec'     : float(np.sum(time_arr))   if len(valid) else None,
}

output = {'per_sample': results, 'stats': stats}
save_results(output, str(RESULTS_DIR / 'marabou_small_mnist.json'))

# ── print summary ─────────────────────────────────────────────────────────────
print()
print('═'*60)
print('  EXACT VERIFICATION RESULTS (SmallMLP)')
print('═'*60)
print(f"  Verifier        : {stats['verifier']}")
print(f"  Samples         : {stats['n_valid']}/{stats['n_samples']} valid")
print(f"  Timeouts        : {stats['n_timeouts']}")
if stats['mean_eps_star'] is not None:
    print(f"  ε* mean ± std   : {stats['mean_eps_star']:.4f} ± {stats['std_eps_star']:.4f}")
    print(f"  ε* median       : {stats['median_eps_star']:.4f}")
    print(f"  Time / sample   : {stats['mean_runtime_sec']:.2f}s")
print('═'*60)
print()
print('  These values serve as GROUND TRUTH for evaluating')
print('  how tight the concolic upper bounds are in Notebook 04.')
print()
print('  Next step → run 04_concolic_exploration.ipynb')

  Saved results → D:\concolic_exploration\results\marabou_small_mnist.json

════════════════════════════════════════════════════════════
  EXACT VERIFICATION RESULTS (SmallMLP)
════════════════════════════════════════════════════════════
  Verifier        : lp_local
  Samples         : 30/30 valid
  Timeouts        : 0
  ε* mean ± std   : 0.1403 ± 0.0517
  ε* median       : 0.1375
  Time / sample   : 0.57s
════════════════════════════════════════════════════════════

  These values serve as GROUND TRUTH for evaluating
  how tight the concolic upper bounds are in Notebook 04.

  Next step → run 04_concolic_exploration.ipynb


## 7 — Scalability timeout experiment (MediumMLP + LargeMLP)

Demonstrates that Marabou/LP verification does not scale to larger networks.
This is an important result for the paper — it justifies why concolic
exploration is needed as a scalable alternative.

We run verification on **5 samples** from each larger model with a short
timeout and record how quickly it fails.

In [8]:
print('Running scalability timeout experiment...')
print('(Verifying 5 samples per model — expected to time out or be very slow)\n')

scalability_results = {}

for model_name, model_path, X_data, y_data in [
    ('medium_mnist', MODELS_DIR / 'medium_mnist.pt', None, None),
    ('large_cifar',  MODELS_DIR / 'large_cifar.pt',  None, None),
]:
    print(f'  {model_name}...')
    m = load_model(str(model_path))

    # load appropriate dataset
    if 'mnist' in model_name:
        X_d, y_d = X_all, y_all   # MNIST already loaded above
    else:
        from torchvision import datasets
        cifar_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                                 (0.2023, 0.1994, 0.2010)),
        ])
        cifar_test = datasets.CIFAR10(DATA_DIR, train=False,
                                      download=True, transform=cifar_tf)
        loader_c   = DataLoader(cifar_test, batch_size=len(cifar_test))
        X_c, y_c   = next(iter(loader_c))
        X_d = X_c.view(X_c.size(0), -1).numpy()
        y_d = y_c.numpy()

    X_s, y_s = get_correctly_classified_samples(m, X_d, y_d, 5, seed=SEED)

    times = []
    for x, label in zip(X_s, y_s):
        t0 = time.time()
        lp_verify_local(m, x, int(label), epsilon=0.01)
        times.append(time.time() - t0)

    mean_t = np.mean(times)
    scalability_results[model_name] = {
        'n_relu_neurons'    : m.n_relu_neurons(),
        'mean_time_per_call': float(mean_t),
        'estimated_full_run': float(mean_t * 100),   # 100 samples
    }
    print(f'    neurons={m.n_relu_neurons()}  '
          f'time/call={mean_t:.2f}s  '
          f'est. 100 samples={mean_t*100:.0f}s ({mean_t*100/60:.1f} min)')

# add small network for comparison
scalability_results['small_mnist'] = {
    'n_relu_neurons'    : model.n_relu_neurons(),
    'mean_time_per_call': stats['mean_runtime_sec'],
    'estimated_full_run': stats['mean_runtime_sec'] * 100 if stats['mean_runtime_sec'] else None,
}

save_results(scalability_results, str(RESULTS_DIR / 'marabou_scalability.json'))

print()
print('Scalability results saved → results/marabou_scalability.json')
print('These will be plotted in Notebook 05.')

Running scalability timeout experiment...
(Verifying 5 samples per model — expected to time out or be very slow)

  medium_mnist...
  Loaded ← D:\concolic_exploration\models\medium_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-256-256-256-10', 'best_test_acc': 0.9853, 'n_relu_neurons': 768, 'n_params': 335114}
    neurons=768  time/call=0.06s  est. 100 samples=6s (0.1 min)
  large_cifar...
  Loaded ← D:\concolic_exploration\models\large_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'architecture': '3072-512-512-512-512-10', 'best_test_acc': 0.5593, 'n_relu_neurons': 2048, 'n_params': 2366474}
    neurons=2048  time/call=0.28s  est. 100 samples=28s (0.5 min)
  Saved results → D:\concolic_exploration\results\marabou_scalability.json

Scalability results saved → results/marabou_scalability.json
These will be plotted in Notebook 05.
